# Clase 42b — De la tabla a las listas · Solucionario

**Solo para el profesor.** No se sube a Classroom ni se proyecta antes de dictar la clase.

Todas las celdas se ejecutan de arriba hacia abajo (scikit-learn, sin archivos).

## ⚙️ Celda preparada del verificador

La misma que ven los estudiantes; los `check_*()` de abajo se ejecutan con las soluciones para comprobar que dan ✅.

In [ ]:
#@title 🔧 Verificador automático — ejecuta esta celda antes de empezar (no la edites)
import base64
import hashlib
import itertools
from sklearn.tree import DecisionTreeClassifier

_NOMBRES = ["dispositivos_conectados", "estudiantes_viendo_video", "horas_desde_reinicio", "temperatura_sala", "hora_del_bloque", "dia_del_mes"]
_TABLA = [[71, 0, 20, 24, 8, 7], [62, 2, 12, 29, 6, 23], [79, 2, 2, 23, 1, 23], [73, 0, 15, 19, 7, 20], [102, 9, 22, 18, 6, 4], [103, 20, 1, 23, 6, 31], [101, 17, 12, 26, 5, 29], [59, 22, 23, 30, 3, 6], [77, 7, 22, 24, 4, 4], [50, 0, 14, 30, 4, 27], [76, 0, 14, 26, 6, 28], [69, 0, 7, 24, 5, 18], [59, 0, 19, 33, 7, 21], [115, 28, 22, 20, 4, 26], [47, 2, 2, 19, 6, 21], [78, 0, 16, 29, 3, 31], [51, 0, 16, 22, 5, 3], [117, 0, 16, 30, 4, 5], [75, 22, 2, 18, 8, 24], [66, 0, 9, 27, 6, 5]]
_ETIQ = [1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0]
_G_TABLA, _G_ETIQ, _G_ACIERTOS = [[[31, 8, 24, 300, 19], [54, 16, 6, 300, 12], [59, 20, 12, 300, 24], [24, 0, 19, 300, 19], [15, 10, 20, 200, 19], [40, 15, 24, 200, 15], [44, 0, 5, 100, 4], [49, 0, 24, 300, 25], [42, 11, 2, 300, 28], [46, 16, 22, 100, 7]], [1, 1, 1, 0, 0, 1, 0, 1, 0, 1], 2]
_E1_TABLA, _E1_ETIQ, _E1_ACIERTOS = [[[62, 18, 10, 24], [50, 5, 0, 12], [43, 3, 0, 15], [30, 18, 3, 31], [32, 2, 0, 25], [17, 12, 16, 31], [19, 0, 13, 12], [15, 7, 16, 22], [24, 0, 6, 30], [60, 10, 15, 21]], [1, 1, 0, 1, 0, 1, 0, 0, 0, 1], 2]
_E2_ETIQ = [1, 1, 0, 0, 1, 0, 0, 0]
_E3_TABLA, _E3_ETIQ = [[[61, 10, 2, 23, 5, 20], [104, 0, 7, 25, 3, 10], [92, 0, 4, 23, 2, 11], [45, 23, 4, 22, 3, 21], [82, 0, 14, 21, 1, 13], [81, 0, 15, 25, 2, 12], [79, 4, 23, 18, 6, 3], [97, 19, 23, 32, 1, 25], [46, 0, 1, 21, 1, 5], [100, 0, 24, 24, 7, 16]], [0, 1, 0, 1, 0, 0, 1, 1, 0, 1]]
_E0A = [[[60, 5, 3], [95, 20, 14], [40, 0, 22]], [0, 1, 0]]


def _columnas_usadas(filas, tabla, inicio):
    """Busca qué columnas (y en qué orden) usó el estudiante: devuelve (columnas, filas que coinciden)."""
    k = len(filas[0])
    mejor, mejor_n = None, -1
    for cols in itertools.permutations(range(len(tabla[0])), k):
        n = sum(1 for i, f in enumerate(filas) if [tabla[inicio + i][c] for c in cols] == list(f))
        if n > mejor_n:
            mejor, mejor_n = cols, n
    return mejor, mejor_n


def _revisar(tabla, etiquetas, n_train, cols_esperadas=None, nombres=None):
    g = globals()
    nombres = nombres or _NOMBRES
    claves = ("entradas_train", "etiquetas_train", "entradas_test", "etiquetas_test")
    valores = [g.get(c) for c in claves]
    faltan = [c for c, v in zip(claves, valores) if not v]
    if faltan:
        print("⏳ Todavía faltan estas listas:", ", ".join(faltan))
        return None
    e_tr, y_tr, e_te, y_te = valores
    n_test = len(tabla) - n_train
    for nombre, v, n in (("entradas_train", e_tr, n_train), ("etiquetas_train", y_tr, n_train),
                         ("entradas_test", e_te, n_test), ("etiquetas_test", y_te, n_test)):
        if len(v) != n:
            print(f"⚠️ {nombre} tiene {len(v)} elementos y debería tener {n} (uno por fila).")
            print("   Pista: cuenta las filas de la tabla y revisa que no falte ninguno ni sobre uno.")
            return None
    for nombre, v in (("entradas_train", e_tr), ("entradas_test", e_te)):
        if not all(isinstance(f, (list, tuple)) and len(f) == len(e_tr[0]) for f in v):
            print(f"⚠️ En {nombre} cada fila debe ser una lista con la misma cantidad de datos.")
            print("   Pista: cada fila va entre corchetes y con una coma entre sus datos.")
            return None
    for nombre, mios, reales, inicio in (("etiquetas_train", y_tr, etiquetas[:n_train], 0),
                                         ("etiquetas_test", y_te, etiquetas[n_train:], n_train)):
        for i, (a, b) in enumerate(zip(mios, reales)):
            if a != b:
                print(f"⚠️ La etiqueta de la posición {i + 1} de {nombre} no coincide con la de la fila {inicio + i + 1} de la tabla.")
                print("   Pista: cada etiqueta debe estar en la misma posición que su fila.")
                return None
    cols, ok = _columnas_usadas(e_tr, tabla, 0)
    if ok < len(e_tr):
        i = next(j for j, f in enumerate(e_tr) if [tabla[j][c] for c in cols] != list(f))
        print(f"⚠️ El fila {i + 1} de entradas_train no coincide con la tabla.")
        print("   Pista: revisa que cada dato esté en la columna correcta y que el orden de las columnas sea el mismo en todas las filas.")
        return None
    cols_te, ok_te = _columnas_usadas(e_te, tabla, n_train)
    if cols_te != cols or ok_te < len(e_te):
        print("⚠️ entradas_test no usa las mismas columnas, en el mismo orden, que entradas_train.")
        print("   Pista: las filas de prueba deben escribirse igual que los de entrenamiento.")
        return None
    if cols_esperadas is not None and tuple(cols) != tuple(cols_esperadas):
        print("⚠️ Tus listas coinciden con la tabla, pero usaste otras columnas: ", ", ".join(nombres[c] for c in cols))
        print("   Pista: las columnas pedidas son:", ", ".join(nombres[c] for c in cols_esperadas), "(en ese orden)")
        return None
    return cols


def _aciertos(cols, tabla, etiquetas, n_train):
    m = DecisionTreeClassifier(max_depth=3, random_state=0)
    m.fit([[f[c] for c in cols] for f in tabla[:n_train]], etiquetas[:n_train])
    pred = m.predict([[f[c] for c in cols] for f in tabla[n_train:]])
    return int(sum(1 for p, y in zip(pred, etiquetas[n_train:]) if p == y))


def _cierre_accuracy(aciertos, n_test):
    ex = globals().get("exactitud")
    if ex is None:
        print("⏳ Falta calcular el accuracy y guardarlo en una variable llamada exactitud.")
    elif abs(ex - aciertos / n_test) < 1e-9:
        print("✅ Tu accuracy es el esperado.")
    else:
        print("⚠️ Tu accuracy no coincide con el esperado.")
        print("   Pista: entrena con profundidad máxima 3 y semilla 0, y compara las predicciones con etiquetas_test.")


def check_icn():
    cols = _revisar(_TABLA, _ETIQ, 16, (0, 1, 2))
    if cols is None:
        return
    print("✅ Tus listas coinciden con la tabla, fila por fila y etiqueta por etiqueta.")
    esperados = [_NOMBRES[c] for c in (0, 1, 2)]
    nv = globals().get("nombres_variables")
    if nv is None:
        print("⏳ Falta escribir nombres_variables con los nombres de tus 3 columnas.")
    elif list(nv) != esperados:
        print("⚠️ Los nombres de nombres_variables no coinciden con tus columnas, o están en otro orden.")
        print("   Pista: deben ir en el mismo orden que las columnas de tus listas:", ", ".join(esperados))
    else:
        print("✅ nombres_variables coincide con tus columnas y con su orden.")
    _cierre_accuracy(_aciertos(cols, _TABLA, _ETIQ, 16), 4)
    print("   Con solo 4 filas de test, ¿alcanza con acertar los 4 para confiar en el modelo?")


def check_guiada():
    cols = _revisar(_G_TABLA, _G_ETIQ, 8, (0, 1, 2), ["computadores_en_uso", "estudiantes_viendo_video",
                                                      "horas_desde_reinicio", "ancho_de_banda_contratado", "dia_del_mes"])
    if cols is None:
        return
    print("✅ Tus listas coinciden con la tabla del liceo vecino.")
    _cierre_accuracy(_G_ACIERTOS, 2)


def check_ej0a():
    e, y = globals().get("entradas"), globals().get("etiquetas")
    if not e or not y:
        print("⏳ Todavía faltan las listas entradas y etiquetas.")
    elif [list(f) for f in e] != _E0A[0]:
        print("⚠️ entradas no coincide con la tabla.")
        print("   Pista: una lista por fila, con los tres datos en el mismo orden de las columnas, todas dentro de otra lista.")
    elif list(y) != _E0A[1]:
        print("⚠️ etiquetas no coincide con la columna cae de la tabla.")
        print("   Pista: una etiqueta por fila, en el mismo orden de las filas.")
    else:
        print("✅ Tus listas coinciden con la tabla.")


def check_ej0b():
    p = globals().get("prediccion")
    m = globals().get("modelo")
    if p is None or m is None:
        print("⏳ Falta guardar la predicción en una variable llamada prediccion.")
    elif list(p) != list(m.predict([[88, 17, 4]])):
        print("⚠️ Tu predicción no corresponde a la fila nueva (88, 17, 4).")
        print("   Pista: una sola fila nueva va dentro de otra lista: [[...]].")
    else:
        print("✅ Predijiste la fila nueva con la forma correcta.")


def check_ej1():
    cols = _revisar(_E1_TABLA, _E1_ETIQ, 8, (0, 1, 2), ["usuarios_conectados", "notebooks_en_mesas",
                                                        "horas_desde_reinicio", "dia_del_mes"])
    if cols is None:
        return
    print("✅ Tus listas coinciden con la tabla de la biblioteca.")
    _cierre_accuracy(_E1_ACIERTOS, 2)


def check_ej2():
    et = globals().get("etiquetas")
    m = globals().get("modelo")
    if et is None:
        print("⏳ Falta la lista etiquetas.")
    elif len(et) != len(_E2_ETIQ):
        print(f"⚠️ etiquetas tiene {len(et)} elementos y hay {len(_E2_ETIQ)} filas.")
        print("   Pista: compara la tabla con tu lista, fila por fila, hasta encontrar la que falta.")
    elif list(et) != _E2_ETIQ:
        print("⚠️ Ya hay una etiqueta por fila, pero alguna quedó en una posición equivocada.")
        print("   Pista: la etiqueta que agregaste debe quedar justo en la posición de su fila, no al final.")
    elif m is None or not hasattr(m, "tree_"):
        print("⏳ Las etiquetas ya están bien. Ejecuta de nuevo la celda completa para que el modelo aprenda.")
    else:
        print("✅ Corregiste la lista: cada etiqueta está en la misma posición que su fila.")


def evaluar(e_tr, y_tr, e_te, y_te):
    """Entrena un árbol (profundidad 3, semilla 0) con train y devuelve el accuracy en test."""
    m = DecisionTreeClassifier(max_depth=3, random_state=0)
    m.fit(e_tr, y_tr)
    pred = m.predict(e_te)
    return sum(1 for p, y in zip(pred, y_te) if p == y) / len(y_te)


def check_ej3():
    cols = _revisar(_E3_TABLA, _E3_ETIQ, 8, None, ["dispositivos_conectados", "estudiantes_viendo_video",
                                                   "horas_desde_reinicio", "temperatura_sala", "hora_del_bloque", "dia_del_mes"])
    if cols is None:
        return
    print("✅ Tus listas coinciden con la tabla. Columnas que usaste:", ", ".join(
        ["dispositivos_conectados", "estudiantes_viendo_video", "horas_desde_reinicio", "temperatura_sala",
         "hora_del_bloque", "dia_del_mes"][c] for c in cols))
    print("   Ahora calcula tu accuracy y el del compañero con evaluar(...) y explica cuál conviene.")


print("Verificador listo ✅")

## 🧠 Haz Ahora

Sin código ni respuesta única: el colega hace de encargado de informática y el curso piensa en voz alta. Esperable: dispositivos conectados, estudiantes viendo video y horas desde el reinicio como las claras; temperatura de la sala y hora del bloque como posibles; el día del mes como la que no tiene nada que ver (se puede comprobar viendo si cambiarla cambia el resultado). **Etiqueta (pregunta 3):** el encargado pregunta «¿se cayó el wifi?» y la respuesta se presenta como un número: **1 = sí, 0 = no** (el 1 significa «sí», no «bueno»). Si el curso propone otra codificación, se aclara que da lo mismo mientras sea binaria y consistente. Filas a guardar para probar: la idea es llegar a «unos pocos, sin mostrarlos al modelo»; el diagrama fija 4 de 20 (20 %). **Tras el Haz Ahora, la clase fija las 3 columnas** (dispositivos, video, horas) y se compara con lo que propuso el curso; si eligieron otras (p. ej. temperatura), se dice que la tabla de comparación de más abajo muestra por qué conviene este trío.

## 📘 ICN — código completo (cada ejemplo es autocontenido)

Trío sensato: `dispositivos_conectados`, `estudiantes_viendo_video`, `horas_desde_reinicio`. Con ese trío el árbol acierta **4 de 4** en el test; ver la tabla comparativa más abajo.

### Concepto 1 · Elegir las 3 columnas (justificación modelo)

**Salida:**

```text

```

La clase ya trae las 3 columnas elegidas; el estudiante solo escribe la justificación. Criterio: nombra las 3 columnas y da una razón para cada grupo (las 3 que sí cargan o desgastan la red; las otras 3 no). Ahorro de procesamiento: menos datos que anotar y escribir, menos ruido y, en bases enormes, menos cómputo.

In [ ]:
# Justificación de ejemplo:
# Elegimos dispositivos_conectados y estudiantes_viendo_video porque son los que más cargan la red;
# y horas_desde_reinicio porque un router que lleva muchas horas sin reiniciarse se degrada.
# Dejamos fuera temperatura_sala, hora_del_bloque y dia_del_mes porque no cargan la red por sí solas.

### Concepto 2 · Una fila es una lista (fila 2)

**Salida:**

```text
Fila 2: [62, 2, 12]
```

In [ ]:
fila_2 = [62, 2, 12]
print("Fila 2:", fila_2)

### Concepto 3 · La tabla de entrenamiento (16 filas)

**Salida:**

```text
Filas de entrenamiento: 16 · etiquetas: 16
```

In [ ]:
fila_2 = [62, 2, 12]

entradas_train = [[71, 0, 20],
    [62, 2, 12],
    [79, 2, 2],
    [73, 0, 15],
    [102, 9, 22],
    [103, 20, 1],
    [101, 17, 12],
    [59, 22, 23],
    [77, 7, 22],
    [50, 0, 14],
    [76, 0, 14],
    [69, 0, 7],
    [59, 0, 19],
    [115, 28, 22],
    [47, 2, 2],
    [78, 0, 16]]
etiquetas_train = [1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1]
print("Filas de entrenamiento:", len(entradas_train), "· etiquetas:", len(etiquetas_train))

### Concepto 4 · Los 4 filas de prueba

**Salida:**

```text
Filas de prueba: 4 · etiquetas: 4
```

In [ ]:
entradas_train = [[71, 0, 20],
    [62, 2, 12],
    [79, 2, 2],
    [73, 0, 15],
    [102, 9, 22],
    [103, 20, 1],
    [101, 17, 12],
    [59, 22, 23],
    [77, 7, 22],
    [50, 0, 14],
    [76, 0, 14],
    [69, 0, 7],
    [59, 0, 19],
    [115, 28, 22],
    [47, 2, 2],
    [78, 0, 16]]
etiquetas_train = [1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1]

entradas_test = [[51, 0, 16],
    [117, 0, 16],
    [75, 22, 2],
    [66, 0, 9]]
etiquetas_test = [0, 1, 1, 0]

print("Filas de prueba:", len(entradas_test), "· etiquetas:", len(etiquetas_test))

### Concepto 5 · La receta: crear → fit → predict

**Salida:**

```text
Predicciones del test: [0 1 1 0]
Etiquetas reales:       [0, 1, 1, 0]
```

`max_depth` = cuánto le dejamos «pensar» al árbol. Verificado con estos datos (entrenamiento): profundidad 1 → se equivoca en 2 de 16; 2 → 1 de 16; 3 → 0 de 16; 4 o más → el árbol queda igual que con 3 (profundidad real 3). Por eso 3 es el menor valor que acierta todo el entrenamiento.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

entradas_train = [[71, 0, 20],
    [62, 2, 12],
    [79, 2, 2],
    [73, 0, 15],
    [102, 9, 22],
    [103, 20, 1],
    [101, 17, 12],
    [59, 22, 23],
    [77, 7, 22],
    [50, 0, 14],
    [76, 0, 14],
    [69, 0, 7],
    [59, 0, 19],
    [115, 28, 22],
    [47, 2, 2],
    [78, 0, 16]]
etiquetas_train = [1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1]

entradas_test = [[51, 0, 16],
    [117, 0, 16],
    [75, 22, 2],
    [66, 0, 9]]
etiquetas_test = [0, 1, 1, 0]

modelo = DecisionTreeClassifier(max_depth=3, random_state=0)
modelo.fit(entradas_train, etiquetas_train)
predicciones = modelo.predict(entradas_test)
print("Predicciones del test:", predicciones)
print("Etiquetas reales:      ", etiquetas_test)

### Concepto 6 · Revisar: leer el árbol y accuracy

**Salida:**

```text
Predicciones del test: [0 1 1 0]
Etiquetas reales:       [0, 1, 1, 0]
|--- estudiantes_viendo_video <= 4.50
|   |--- horas_desde_reinicio <= 15.50
|   |   |--- class: 0
|   |--- horas_desde_reinicio >  15.50
|   |   |--- dispositivos_conectados <= 65.00
|   |   |   |--- class: 0
|   |   |--- dispositivos_conectados >  65.00
|   |   |   |--- class: 1
|--- estudiantes_viendo_video >  4.50
|   |--- class: 1

Accuracy en el test: 1.0
```

Lectura del árbol: la primera pregunta es `estudiantes_viendo_video <= 4,5`; si no (más de 4,5 estudiantes viendo video), predice 1 (sí se cae). Si sí, mira `horas_desde_reinicio <= 15,5`: con pocas horas desde el reinicio predice 0; con más, mira `dispositivos_conectados <= 65` (0) o más (1). Las **features** son las 3 columnas que le dimos; los nombres deben ir en el mismo orden que las columnas de las listas. Accuracy 4 de 4 = 1,0: el colega puede preguntar si eso basta para confiar (no basta: son solo 4 filas).

In [ ]:
from sklearn.tree import export_text
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

entradas_train = [[71, 0, 20],
    [62, 2, 12],
    [79, 2, 2],
    [73, 0, 15],
    [102, 9, 22],
    [103, 20, 1],
    [101, 17, 12],
    [59, 22, 23],
    [77, 7, 22],
    [50, 0, 14],
    [76, 0, 14],
    [69, 0, 7],
    [59, 0, 19],
    [115, 28, 22],
    [47, 2, 2],
    [78, 0, 16]]
etiquetas_train = [1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1]

entradas_test = [[51, 0, 16],
    [117, 0, 16],
    [75, 22, 2],
    [66, 0, 9]]
etiquetas_test = [0, 1, 1, 0]

modelo = DecisionTreeClassifier(max_depth=3, random_state=0)
modelo.fit(entradas_train, etiquetas_train)
predicciones = modelo.predict(entradas_test)
print("Predicciones del test:", predicciones)
print("Etiquetas reales:      ", etiquetas_test)

# Revisar 1: leer el árbol (los nombres van en el MISMO orden que las columnas de mis listas)
nombres_variables = ['dispositivos_conectados', 'estudiantes_viendo_video', 'horas_desde_reinicio']
print(export_text(modelo, feature_names=nombres_variables))

# Revisar 2: precisión (accuracy)
exactitud = accuracy_score(etiquetas_test, predicciones)
print("Accuracy en el test:", exactitud)

In [ ]:
check_icn()

### 📊 Comparación de tríos (verificado con scikit-learn 1.9, profundidad 3, semilla 0)

| Trío | Aciertos en test (de 4) | Validación cruzada |
|---|---|---|
| **Sensato** (dispositivos, video, horas reinicio) | **4** | **0,90** (el mejor de los 20 tríos) |
| dispositivos, video, temperatura | 4 | 0,70 |
| dispositivos, video, día del mes (con la columna sin relación) | 2 | 0,80 |
| dispositivos, temperatura, día del mes | 1 | 0,55 |
| temperatura, hora del bloque, día del mes (los tres «flojos») | 1 | 0,55 |

Lectura honesta: con solo 4 filas de test, un trío con una columna floja puede sacar 4/4 igual que el sensato; lo que lo delata es la validación cruzada (que no se enseña). El árbol con las 6 columnas usa solo las 3 claras.

## 🧩 Práctica guiada — solución

### Guiada · El liceo vecino

**Salida:**

```text
Predicciones de las 2 filas de prueba: [0 1]
Etiquetas reales: [0, 1]
Accuracy en el test: 1.0
```

Columnas pedidas: las tres primeras. Con 2 filas de prueba, acertar los 2 no basta para confiar; ese es el comentario esperado. **Revisión en vivo:** una pareja explica al frente cómo pasó cada fila de la tabla a su lista y por qué cada etiqueta queda en su posición.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

entradas_train = [[31, 8, 24],
    [54, 16, 6],
    [59, 20, 12],
    [24, 0, 19],
    [15, 10, 20],
    [40, 15, 24],
    [44, 0, 5],
    [49, 0, 24]]
etiquetas_train = [1, 1, 1, 0, 0, 1, 0, 1]

entradas_test = [[42, 11, 2],
    [46, 16, 22]]
etiquetas_test = [0, 1]

modelo = DecisionTreeClassifier(max_depth=3, random_state=0)
modelo.fit(entradas_train, etiquetas_train)
predicciones = modelo.predict(entradas_test)
exactitud = accuracy_score(etiquetas_test, predicciones)

print("Predicciones de las 2 filas de prueba:", predicciones)
print("Etiquetas reales:", etiquetas_test)
print("Accuracy en el test:", exactitud)

# Con solo 2 filas de prueba, acertar los 2 no basta para confiar: conviene probar con más filas.

In [ ]:
check_guiada()

## 🚀 Ejercicios extra — soluciones

### Ejercicio 0a

**Salida:**

```text
entradas: [[60, 5, 3], [95, 20, 14], [40, 0, 22]]
etiquetas: [0, 1, 0]
```

In [ ]:
entradas = [[60, 5, 3], [95, 20, 14], [40, 0, 22]]
etiquetas = [0, 1, 0]
print("entradas:", entradas)
print("etiquetas:", etiquetas)

In [ ]:
check_ej0a()

### Ejercicio 0b

**Salida:**

```text
Predicción para la fila nueva (1 = se cae, 0 = aguanta): [0]
```

In [ ]:
from sklearn.tree import DecisionTreeClassifier

entradas = [[60, 5, 3],
    [95, 20, 14],
    [40, 0, 22],
    [110, 2, 8],
    [30, 12, 5],
    [85, 0, 18]]
etiquetas = [0, 1, 0, 1, 1, 0]

modelo = DecisionTreeClassifier(max_depth=2, random_state=0)
modelo.fit(entradas, etiquetas)

prediccion = modelo.predict([[88, 17, 4]])
print("Predicción para la fila nueva (1 = se cae, 0 = aguanta):", prediccion)

In [ ]:
check_ej0b()

### Ejercicio 1 · Biblioteca

**Salida:**

```text
Predicciones del test: [0 1]
Accuracy en el test: 1.0
```

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

entradas_train = [[62, 18, 10],
    [50, 5, 0],
    [43, 3, 0],
    [30, 18, 3],
    [32, 2, 0],
    [17, 12, 16],
    [19, 0, 13],
    [15, 7, 16]]
etiquetas_train = [1, 1, 0, 1, 0, 1, 0, 0]

entradas_test = [[24, 0, 6],
    [60, 10, 15]]
etiquetas_test = [0, 1]

modelo = DecisionTreeClassifier(max_depth=3, random_state=0)
modelo.fit(entradas_train, etiquetas_train)
predicciones = modelo.predict(entradas_test)
exactitud = accuracy_score(etiquetas_test, predicciones)
print("Predicciones del test:", predicciones)
print("Accuracy en el test:", exactitud)

In [ ]:
check_ej1()

### Ejercicio 2 · Sala de computación (la etiqueta que falta)

**Salida:**

```text
El modelo ya aprendió de 8 filas
```

La lista del enunciado traía `[1, 1, 0, 0, 0, 0, 0]` (falta la 5ª etiqueta, que es `1`). El error al entrenar es un `ValueError` por distinta cantidad de entradas y etiquetas.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

entradas = [[36, 7, 2],
    [29, 14, 21],
    [25, 9, 0],
    [18, 2, 13],
    [19, 12, 2],
    [11, 8, 7],
    [12, 9, 12],
    [10, 0, 10]]
etiquetas = [1, 1, 0, 0, 1, 0, 0, 0]

modelo = DecisionTreeClassifier(max_depth=3, random_state=0)
modelo.fit(entradas, etiquetas)
print("El modelo ya aprendió de", len(entradas), "filas")

In [ ]:
check_ej2()

### Ejercicio 3 · Sala de actos (comparar tríos)

**Salida:**

```text
Accuracy de mi trío: 1.0
Accuracy del trío del compañero: 0.0
```

Criterio: elige tres columnas con razones, calcula los dos accuracy y argumenta; debe notar que con 2 filas de prueba una diferencia pequeña prueba poco. El trío sensato acierta 2 de 2 y el del compañero (`temperatura_sala`, `hora_del_bloque`, `dia_del_mes`) 0 de 2.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Mi trío: dispositivos_conectados, estudiantes_viendo_video, horas_desde_reinicio
entradas_train = [[61, 10, 2],
    [104, 0, 7],
    [92, 0, 4],
    [45, 23, 4],
    [82, 0, 14],
    [81, 0, 15],
    [79, 4, 23],
    [97, 19, 23]]
etiquetas_train = [0, 1, 0, 1, 0, 0, 1, 1]

entradas_test = [[46, 0, 1],
    [100, 0, 24]]
etiquetas_test = [0, 1]

def evaluar(e_tr, y_tr, e_te, y_te):
    m = DecisionTreeClassifier(max_depth=3, random_state=0)
    m.fit(e_tr, y_tr)
    return accuracy_score(y_te, m.predict(e_te))

# Trío del compañero: temperatura_sala, hora_del_bloque, dia_del_mes
entradas_train_companero = [[23, 5, 20],
    [25, 3, 10],
    [23, 2, 11],
    [22, 3, 21],
    [21, 1, 13],
    [25, 2, 12],
    [18, 6, 3],
    [32, 1, 25]]
etiquetas_train_companero = [0, 1, 0, 1, 0, 0, 1, 1]

entradas_test_companero = [[21, 1, 5],
    [24, 7, 16]]
etiquetas_test_companero = [0, 1]

mi_accuracy = evaluar(entradas_train, etiquetas_train, entradas_test, etiquetas_test)
su_accuracy = evaluar(entradas_train_companero, etiquetas_train_companero, entradas_test_companero, etiquetas_test_companero)
print("Accuracy de mi trío:", mi_accuracy)
print("Accuracy del trío del compañero:", su_accuracy)

In [ ]:
check_ej3()

### Ejercicio 4 · Desafío

**Salida:**

```text
Accuracy de mi trío: 1.0
Accuracy del trío del compañero: 0.0
Accuracy de mi trío con el dato mal anotado: 0.5
```

Con el trío sensato, un solo dato mal anotado (dispositivos = 100 en la primera fila de prueba) baja el accuracy de 1,0 a 0,5. Criterio: nota que cambió, y lo relaciona con que el test es muy chico (una sola fila pesa la mitad) y con la importancia de revisar los datos con atención.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Mi trío: dispositivos_conectados, estudiantes_viendo_video, horas_desde_reinicio
entradas_train = [[61, 10, 2],
    [104, 0, 7],
    [92, 0, 4],
    [45, 23, 4],
    [82, 0, 14],
    [81, 0, 15],
    [79, 4, 23],
    [97, 19, 23]]
etiquetas_train = [0, 1, 0, 1, 0, 0, 1, 1]

entradas_test = [[46, 0, 1],
    [100, 0, 24]]
etiquetas_test = [0, 1]

def evaluar(e_tr, y_tr, e_te, y_te):
    m = DecisionTreeClassifier(max_depth=3, random_state=0)
    m.fit(e_tr, y_tr)
    return accuracy_score(y_te, m.predict(e_te))

# Trío del compañero: temperatura_sala, hora_del_bloque, dia_del_mes
entradas_train_companero = [[23, 5, 20],
    [25, 3, 10],
    [23, 2, 11],
    [22, 3, 21],
    [21, 1, 13],
    [25, 2, 12],
    [18, 6, 3],
    [32, 1, 25]]
etiquetas_train_companero = [0, 1, 0, 1, 0, 0, 1, 1]

entradas_test_companero = [[21, 1, 5],
    [24, 7, 16]]
etiquetas_test_companero = [0, 1]

mi_accuracy = evaluar(entradas_train, etiquetas_train, entradas_test, etiquetas_test)
su_accuracy = evaluar(entradas_train_companero, etiquetas_train_companero, entradas_test_companero, etiquetas_test_companero)
print("Accuracy de mi trío:", mi_accuracy)
print("Accuracy del trío del compañero:", su_accuracy)

# Cambio: el primer dato de la primera fila de prueba quedó mal anotado y pasa a valer 100
entradas_test[0][0] = 100
print("Accuracy de mi trío con el dato mal anotado:", evaluar(entradas_train, etiquetas_train, entradas_test, etiquetas_test))

## 🎫 Ticket de salida — preguntas y respuestas

Respuestas correctas: **D, B, C, A**. Las preguntas **no están en el Colab de estudiantes**: se proyectan en `Ticket de Salida.pptx` y se responden en el Google Form (`https://forms.gle/sjRpbgmQzrpkEBsH9`). Las respuestas correctas viven también en `Ticket de Salida Respuestas.json`.

### Pregunta 1 — La forma de la tabla

Tres filas del wifi, con tres datos cada uno (dispositivos, estudiantes viendo video, horas desde el reinicio):

```text
Fila 1: 71, 0, 20
Fila 2: 62, 2, 12
Fila 3: 79, 2, 2
```

¿Cuál de estas opciones escribe correctamente la tabla de entradas?

- **A)** `[71, 0, 20, 62, 2, 12, 79, 2, 2]`
- **B)** `[[71, 62, 79], [0, 2, 2], [20, 12, 2]]`
- **C)** `[[71, 0, 20] [62, 2, 12] [79, 2, 2]]`
- **D)** `[[71, 0, 20], [62, 2, 12], [79, 2, 2]]`

**Respuesta correcta: D.** Cada fila va en su propia lista, las filas se separan con coma y todo va dentro de otro par de corchetes.

### Pregunta 2 — Cada etiqueta con su fila

```python
entradas = [[71, 0, 20], [62, 2, 12], [79, 2, 2], [73, 0, 15]]
etiquetas = [1, 1, 0, 1]
```

¿Qué etiqueta le corresponde a la fila 3, `[79, 2, 2]`?

- **A)** `1`, la de la última fila.
- **B)** `0`, la que está en la misma posición que la fila 3.
- **C)** `1`, la de la fila 2.
- **D)** No se puede saber, porque las etiquetas están en otra lista.

**Respuesta correcta: B.** Cada etiqueta está en la misma posición que su fila: la fila 3 se empareja con la tercera etiqueta, que es 0.

### Pregunta 3 — Predecir una sola fila

```python
modelo.fit(entradas_train, etiquetas_train)
resultado = modelo.predict(...)   # <- completa esta línea
```

Queremos preguntar por **una sola** fila nueva: 88 dispositivos, 17 estudiantes viendo video y 4 horas desde el reinicio. ¿Cómo se escribe?

- **A)** `modelo.predict([88, 17, 4])`
- **B)** `modelo.predict(88, 17, 4)`
- **C)** `modelo.predict([[88, 17, 4]])`
- **D)** `modelo.predict([[88], [17], [4]])`

**Respuesta correcta: C.** predict() espera una tabla (una lista de filas); una sola fila nueva va dentro de otra lista: [[88, 17, 4]].

### Pregunta 4 — Pensar sobre el test

Un modelo se prueba con las 4 filas que guardamos aparte y acierta los 4 (accuracy = 1,0).

¿Qué conviene concluir?

- **A)** Que acertó esas 4 filas; con tan pocos no basta para confiar, conviene probarlo con más.
- **B)** Que el modelo es perfecto y se puede usar sin revisar.
- **C)** Que el test estaba mal elegido, porque un modelo nunca debería acertar todo.
- **D)** Que el modelo se aprendió de memoria las 4 filas del test.

**Respuesta correcta: A.** Acertó esas 4 filas, pero con tan pocos no basta para confiar: conviene probar el modelo con más filas.